# HW03 2.1 理论计算题

---

### 题目1：卷积层输出特征图尺寸计算
**题目**：输入一张大小为 $3 \times 32 \times 32$（通道数 × 高 × 宽）的彩色图像。通过一个卷积层，该层包含16个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$。设定填充（Padding）为2，步幅（Stride）为2。请计算该卷积层输出的特征图（Feature Map）的尺寸（通道数 × 高 × 宽）。

**解答**：
1.  **参数整理**
    - 输入通道数 $C_{in} = 3$，输入高 $H_{in}=32$，输入宽 $W_{in}=32$
    - 卷积核数量 $K=16$（即输出通道数 $C_{out}=16$）
    - 卷积核空间尺寸 $k_h=5, k_w=5$，填充 $P=2$，步幅 $S=2$

2.  **空间尺寸计算公式**
    卷积层输出特征图的高和宽计算公式为：
    $$
    H_{out} = \left\lfloor \frac{H_{in} + 2P - k_h}{S} \right\rfloor + 1
    $$
    $$
    W_{out} = \left\lfloor \frac{W_{in} + 2P - k_w}{S} \right\rfloor + 1
    $$

3.  **代入计算**
    $$
    H_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
    $$
    $$
    W_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = 16
    $$

4.  **最终结果**
    输出特征图尺寸为：$\boldsymbol{16 \times 16 \times 16}$（通道数 × 高 × 宽）。

---

### 题目2：卷积操作点乘次数计算
**题目**：计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行多少次点乘（乘法）操作？

**解答**：
1.  **点乘次数的本质**
    卷积操作中，单个输出像素值是输入特征图与卷积核的对应元素逐点相乘后求和。因此，点乘次数等于卷积核的总参数数量。

2.  **计算过程**
    卷积核的总参数数量 = 输入通道数 × 卷积核高 × 卷积核宽
    代入参数：$3 \times 5 \times 5 = 75$

3.  **最终结果**
    单个输出通道的一个像素值，需要进行 $\boldsymbol{75}$ 次点乘（乘法）操作。

In [1]:
# 2.2
import numpy as np

def max_pool2d_forward(x, kernel_size, stride, padding):
    """
    手动实现二维最大池化前向传播
    参数:
        x: 输入特征图，形状为 (N, C, H, W) （批次×通道×高×宽）
        kernel_size: 池化核大小，int 或 tuple (k_h, k_w)
        stride: 步幅，int 或 tuple (s_h, s_w)
        padding: 填充，int 或 tuple (p_h, p_w)
    返回:
        out: 池化输出，形状为 (N, C, H_out, W_out)
    """
    # 统一参数格式
    if isinstance(kernel_size, int):
        k_h, k_w = kernel_size, kernel_size
    else:
        k_h, k_w = kernel_size
    if isinstance(stride, int):
        s_h, s_w = stride, stride
    else:
        s_h, s_w = stride
    if isinstance(padding, int):
        p_h, p_w = padding, padding
    else:
        p_h, p_w = padding

    N, C, H, W = x.shape

    # 1. 对输入进行填充
    x_pad = np.pad(x, ((0, 0), (0, 0), (p_h, p_h), (p_w, p_w)), mode='constant', constant_values=0)

    # 2. 计算输出尺寸
    H_out = (H + 2 * p_h - k_h) // s_h + 1
    W_out = (W + 2 * p_w - k_w) // s_w + 1

    # 3. 初始化输出
    out = np.zeros((N, C, H_out, W_out))

    # 4. 遍历每个池化窗口，计算最大值
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    # 窗口在填充后特征图中的位置
                    h_start = i * s_h
                    h_end = h_start + k_h
                    w_start = j * s_w
                    w_end = w_start + k_w
                    # 取窗口内的最大值
                    window = x_pad[n, c, h_start:h_end, w_start:w_end]
                    out[n, c, i, j] = np.max(window)

    return out


# 测试代码
if __name__ == "__main__":
    # 构造测试输入 (N=1, C=1, H=4, W=4)
    x = np.array([
        [[
            [1, 2, 3, 4],
            [5, 6, 7, 8],
            [9, 10, 11, 12],
            [13, 14, 15, 16]
        ]]
    ], dtype=np.float32)

    # 池化参数
    kernel_size = 2
    stride = 2
    padding = 0

    # 手动实现的池化
    out_custom = max_pool2d_forward(x, kernel_size, stride, padding)
    print("手动实现MaxPool2d输出：")
    print(out_custom)

手动实现MaxPool2d输出：
[[[[ 6.  8.]
   [14. 16.]]]]


# 3.1
---

### 题目1：计算一个 $5 \times 5$ 卷积层（不带偏置）的参数量

**题目描述**：
假设输入和输出的特征图通道数均为 $C$，计算一个 $5 \times 5$ 卷积层（不带偏置）的参数量。

**解答**：
1.  **参数量计算公式（无偏置）**
    卷积层的参数量由输入通道数、输出通道数和卷积核尺寸决定，公式为：
    $$
    \text{参数量} = \text{输入通道数} \times \text{输出通道数} \times \text{卷积核高} \times \text{卷积核宽}
    $$

2.  **代入参数计算**
    本题中，输入通道数为 $C$，输出通道数为 $C$，卷积核尺寸为 $5 \times 5$，因此：
    $$
    \text{参数量} = C \times C \times 5 \times 5 = 25C^2
    $$

3.  **最终结果**
    该 $5 \times 5$ 卷积层（不带偏置）的参数量为：$\boldsymbol{25C^2}$。

---

### 题目2：计算两个串联的 $3 \times 3$ 卷积层（不带偏置）的总参数量

**题目描述**：
计算两个串联的 $3 \times 3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量。

**解答**：
1.  **单个 $3 \times 3$ 卷积层的参数量**
    对于单个 $3 \times 3$ 卷积层，输入通道数为 $C$，输出通道数为 $C$，无偏置时的参数量为：
    $$
    \text{单层参数量} = C \times C \times 3 \times 3 = 9C^2
    $$

2.  **两个串联卷积层的总参数量**
    两个串联的 $3 \times 3$ 卷积层，通道数均为 $C$，因此总参数量为两层参数量之和：
    $$
    \text{总参数量} = 9C^2 + 9C^2 = 18C^2
    $$

3.  **最终结果**
    两个串联的 $3 \times 3$ 卷积层（不带偏置）的总参数量为：$\boldsymbol{18C^2}$。

In [10]:
#3.2

import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    标准 NiN 块实现，结构为：
    普通卷积层 → ReLU → 1×1卷积层 → ReLU → 1×1卷积层 → ReLU
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        # 使用 nn.Sequential 定义 NiN 块结构
        self.block = nn.Sequential(
            # 第一层：普通卷积层（带ReLU）
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding
            ),
            nn.ReLU(inplace=True),
            
            # 第二层：1×1卷积层（带ReLU）
            nn.Conv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,
                padding=0
            ),
            nn.ReLU(inplace=True),
            
            # 第三层：1×1卷积层（带ReLU）
            nn.Conv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,
                padding=0
            ),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 示例：定义一个 NiN 块
    # 输入通道3，输出通道16，卷积核3×3，步幅1，填充1
    nin_block = NiNBlock(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        stride=1,
        padding=1
    )

    # 构造随机输入张量（batch_size=2, 通道=3, 高=32, 宽=32）
    x = torch.randn(2, 3, 32, 32)
    # 前向传播
    output = nin_block(x)

    print("输入形状:", x.shape)    
    print("输出形状:", output.shape) 

输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 16, 32, 32])


# 4.1 

---

### 题目：Batch Normalization 层输出计算
**题目描述**：  
在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$。假设当前批量归一化层学到的缩放参数 $\gamma = 2$，平移参数 $\beta = 1$，常数 $\epsilon = 0$。请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 $y_1, y_2, y_3, y_4$。

**解答**：

#### 1. 批量归一化公式
批量归一化的计算流程为：
$$
y_i = \gamma \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$
其中：
- $\mu$：批量均值
- $\sigma^2$：批量方差
- $\gamma$：缩放参数
- $\beta$：平移参数
- $\epsilon$：数值稳定常数（防止分母为0）

---

#### 2. 步骤1：计算批量均值 $\mu$
$$
\mu = \frac{1}{N} \sum_{i=1}^N x_i = \frac{2 + 4 + 6 + 8}{4} = 5
$$

---

#### 3. 步骤2：计算批量方差 $\sigma^2$
$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^N (x_i - \mu)^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = 5
$$

---

#### 4. 步骤3：计算标准差 $\sigma$
由于 $\epsilon = 0$，标准差为：
$$
\sigma = \sqrt{\sigma^2 + \epsilon} = \sqrt{5}
$$

---

#### 5. 步骤4：代入公式计算每个样本的输出值
$$
\begin{align*}
y_1 &= 2 \cdot \frac{2 - 5}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}} = 1 - \frac{6\sqrt{5}}{5} \approx -1.683 \\
y_2 &= 2 \cdot \frac{4 - 5}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}} = 1 - \frac{2\sqrt{5}}{5} \approx 0.106 \\
y_3 &= 2 \cdot \frac{6 - 5}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}} = 1 + \frac{2\sqrt{5}}{5} \approx 1.894 \\
y_4 &= 2 \cdot \frac{8 - 5}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}} = 1 + \frac{6\sqrt{5}}{5} \approx 3.683
\end{align*}
$$

---

#### 6. 最终结果
| 样本 | 原始值 $x_i$ | 归一化后输出 $y_i$（精确形式） | 归一化后输出 $y_i$（近似值） |
| :--- | :--- | :--- | :--- |
| 1 | 2 | $1 - \frac{6\sqrt{5}}{5}$ | -1.683 |
| 2 | 4 | $1 - \frac{2\sqrt{5}}{5}$ | 0.106 |
| 3 | 6 | $1 + \frac{2\sqrt{5}}{5}$ | 1.894 |
| 4 | 8 | $1 + \frac{6\sqrt{5}}{5}$ | 3.683 |

---

**结论**：4个样本经过Batch Normalization层后的输出值分别为：
$$
\boldsymbol{y_1 = 1 - \frac{6\sqrt{5}}{5},\ y_2 = 1 - \frac{2\sqrt{5}}{5},\ y_3 = 1 + \frac{2\sqrt{5}}{5},\ y_4 = 1 + \frac{6\sqrt{5}}{5}}
$$
（或近似为 $-1.683, 0.106, 1.894, 3.683$）。

In [4]:
#4.2
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    标准 ResNet 残差块实现
    结构：Conv3x3 → BN → ReLU → Conv3x3 → BN → (残差连接) → ReLU
    支持通过 use_1x1conv 调整输入通道/尺寸
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super(Residual, self).__init__()
        
        # 主路径：两个 3×3 卷积层，每层后接 BatchNorm
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=stride,
            padding=1  # 3×3卷积 + padding=1，保证stride=1时尺寸不变
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1×1卷积：用于调整输入通道/尺寸（当通道或步幅变化时）
        if use_1x1conv:
            self.conv1x1 = nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=stride  # 步幅与主路径一致，保证空间尺寸匹配
            )
        else:
            self.conv1x1 = None

    def forward(self, x):
        # 主路径前向传播
        identity = x  # 保存原始输入作为残差
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 残差连接：如果需要调整，先对输入进行1×1卷积
        if self.conv1x1 is not None:
            identity = self.conv1x1(x)
        
        # 残差相加
        out += identity
        out = self.relu(out)
        
        return out


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 测试场景1：通道不变、步幅1（基础残差块）
    print("测试场景1：通道不变、步幅1")
    residual1 = Residual(in_channels=64, out_channels=64, stride=1, use_1x1conv=False)
    x1 = torch.randn(2, 64, 32, 32)  # batch=2, 通道=64, 尺寸=32×32
    output1 = residual1(x1)
    print(f"输入形状: {x1.shape}")
    print(f"输出形状: {output1.shape}\n")

    # 测试场景2：通道变化、步幅2（带1×1卷积的残差块）
    print("测试场景2：通道变化、步幅2（带1×1卷积）")
    residual2 = Residual(in_channels=64, out_channels=128, stride=2, use_1x1conv=True)
    x2 = torch.randn(2, 64, 32, 32)
    output2 = residual2(x2)
    print(f"输入形状: {x2.shape}")
    print(f"输出形状: {output2.shape}")

测试场景1：通道不变、步幅1
输入形状: torch.Size([2, 64, 32, 32])
输出形状: torch.Size([2, 64, 32, 32])

测试场景2：通道变化、步幅2（带1×1卷积）
输入形状: torch.Size([2, 64, 32, 32])
输出形状: torch.Size([2, 128, 16, 16])


# 5.1 

---

### 题目1：微调时学习率设置差异的原因
**题目**：为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？

**解答**：
这种设置是由预训练模型不同层学到的特征性质差异决定的，核心原因如下：

1.  **底层特征的通用性与稳定性**
    预训练模型的底层卷积层学习的是通用视觉特征（如边缘、纹理、颜色、简单形状等），这些特征在ImageNet等大型数据集上学习到，对绝大多数视觉任务都具有通用性。大幅修改这些层的参数会破坏模型已学到的通用特征，反而降低泛化能力，因此需要用小学习率微调，甚至直接冻结。

2.  **防止灾难性遗忘**
    底层特征提取层的预训练权重已经非常稳定，使用大学习率更新会导致模型快速遗忘已学到的通用知识，这种现象称为“灾难性遗忘”。小学习率可以保证模型在不破坏旧知识的前提下，缓慢适应新任务。

3.  **顶层输出层的任务特异性**
    预训练模型的顶层输出层是针对源数据集（如ImageNet的1000类分类）设计的，与目标任务不匹配，因此通常会被替换为新初始化的层。这些层没有预训练权重，需要通过大学习率快速学习目标任务的特异性特征（如目标类别的决策边界），以实现模型的快速收敛。

4.  **梯度更新的安全性**
    底层特征的梯度更新会反向影响整个网络，而顶层的梯度更新仅作用于新初始化的层，不会破坏预训练的底层特征。因此，顶层可以使用较大的学习率，而底层需要严格控制更新幅度。

---

### 题目2：小而相似的目标数据集的防过拟合微调策略
**题目**：如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？

**解答**：
针对“数据集小、与源数据集高度相似”的场景，核心思路是**最大限度复用预训练模型的通用特征，减少可训练参数，降低模型复杂度**，具体策略如下：

1.  **冻结大部分底层，仅微调顶层**
    由于目标数据集与源数据集高度相似，预训练模型的底层特征完全可以直接复用。因此可以冻结大部分卷积层，仅微调最顶层的分类层或少量全连接层，大幅减少可训练参数，降低过拟合风险。

2.  **使用极低的学习率，甚至仅训练分类层**
    进一步降低微调的学习率（如预训练阶段学习率的1/10~1/100），甚至直接将预训练模型作为固定的特征提取器，仅在提取的特征上训练一个简单的线性分类器，完全避免更新预训练参数，从根源上防止过拟合。

3.  **添加正则化约束**
    在顶层分类层添加Dropout层、L2权重衰减（Weight Decay）等正则化技术，限制模型的复杂度，防止模型过度拟合训练数据中的噪声。

4.  **使用数据增强扩充目标数据集**
    对目标数据集进行轻量级图像增广（如随机翻转、裁剪、亮度/对比度调整等），增加数据多样性，提升模型的泛化能力，缓解数据量不足的问题。

5.  **引入早停（Early Stopping）机制**
    监控验证集的损失，当验证集损失不再下降时立即停止训练，防止模型在小数据集上过度迭代，避免过拟合。

6.  **采用更简单的微调范式**
    如使用线性探测（Linear Probing）：冻结整个预训练模型，仅训练一个线性分类器；或使用低秩适配（LoRA）等轻量级微调技术，仅调整模型的部分参数，大幅降低过拟合风险。

In [11]:
#5.2
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# 定义图像增广管道（Pipeline）
train_transform = transforms.Compose([
    # 1. 随机裁剪并缩放到224×224像素，面积比例在0.08~1.0之间
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.08, 1.0)
    ),
    # 2. 以50%的概率对图像进行水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机调整亮度、对比度和饱和度，变化范围设为0.5
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),
    # 4. 将PIL图像转换为PyTorch张量
    transforms.ToTensor()
])


# 测试代码
if __name__ == "__main__":
    # 1. 生成一个随机的测试图像（256×256，3通道）
    # 用numpy生成随机像素值，转成PIL图像
    np_img = np.random.randint(0, 256, size=(256, 256, 3), dtype=np.uint8)
    img = Image.fromarray(np_img)
    
    # 2. 应用图像增广管道
    transformed_img = train_transform(img)
    
    # 3. 输出结果验证
    print("增广后图像张量形状:", transformed_img.shape)
    print("   预期输出: torch.Size([3, 224, 224])")
    


增广后图像张量形状: torch.Size([3, 224, 224])
   预期输出: torch.Size([3, 224, 224])


# 6.1 

---

### 题目：边界框 IoU 计算
**题目描述**：  
在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知两个边界框（格式为`[左上角x, 左上角y, 右下角x, 右下角y]`）：
真实框（Ground Truth）$A = [10, 10, 50, 50]$
预测框（Prediction Box）$B = [30, 30, 70, 70]$

请计算边界框 $A$ 和 $B$ 之间的 IoU 准确值。

**解答**：

#### 1. IoU 定义
交并比（Intersection over Union, IoU）的计算公式为：
$$
\text{IoU} = \frac{\text{交集面积}}{\text{并集面积}}
$$

---

#### 2. 步骤1：计算单个边界框的面积
真实框 $A$ 的宽高：$w_A = 50 - 10 = 40$，$h_A = 50 - 10 = 40$  
  面积：$Area_A = w_A \times h_A = 40 \times 40 = 1600$

预测框 $B$ 的宽高：$w_B = 70 - 30 = 40$，$h_B = 70 - 30 = 40$  
  面积：$Area_B = w_B \times h_B = 40 \times 40 = 1600$

---

#### 3. 步骤2：计算交集的坐标与面积
交集框的坐标计算规则：
 交集左上角 $x$：$\max(A_{x1}, B_{x1}) = \max(10, 30) = 30$
 交集左上角 $y$：$\max(A_{y1}, B_{y1}) = \max(10, 30) = 30$
 交集右下角 $x$：$\min(A_{x2}, B_{x2}) = \min(50, 70) = 50$
 交集右下角 $y$：$\min(A_{y2}, B_{y2}) = \min(50, 70) = 50$

交集框为 $[30, 30, 50, 50]$，其宽高为：
$w_{inter} = 50 - 30 = 20$，$h_{inter} = 50 - 30 = 20$

交集面积：
$$
Area_{inter} = w_{inter} \times h_{inter} = 20 \times 20 = 400
$$

---

#### 4. 步骤3：计算并集面积
并集面积公式：
$$
Area_{union} = Area_A + Area_B - Area_{inter}
$$

代入数值：
$$
Area_{union} = 1600 + 1600 - 400 = 2800
$$

---

#### 5. 步骤4：计算 IoU
$$
\text{IoU} = \frac{Area_{inter}}{Area_{union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429
$$

---

#### 最终结果
边界框 $A$ 和 $B$ 之间的 IoU 准确值为：
$$
\boldsymbol{\text{IoU} = \frac{1}{7} \approx 0.1429}
$$

In [12]:
#6.2
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, reduction='mean'):
    """
    实现带标签平滑的交叉熵损失函数
    
    参数:
        logits: 模型的原始输出（未经过softmax），形状为 [batch_size, num_classes]
        labels: 真实标签（类别的索引），形状为 [batch_size]
        epsilon: 标签平滑因子，题目中为0.1
        reduction: 损失的聚合方式，可选 'mean'/'sum'/'none'
    
    返回:
        计算得到的损失值
    """
    # 获取批次大小和类别数
    batch_size, num_classes = logits.shape
    
    # 1. 将真实标签转换为one-hot编码
    one_hot = torch.zeros_like(logits).scatter(1, labels.unsqueeze(1), 1)
    
    # 2. 计算标签平滑后的目标分布
    # 真实类别: 1 - epsilon，其他类别: epsilon / (num_classes - 1)
    smooth_labels = one_hot * (1 - epsilon) + (1 - one_hot) * epsilon / (num_classes - 1)
    
    # 3. 对模型输出进行log_softmax（数值更稳定）
    log_probs = F.log_softmax(logits, dim=1)
    
    # 4. 计算交叉熵损失
    loss = - (smooth_labels * log_probs).sum(dim=1)
    
    # 5. 根据reduction参数聚合损失
    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 测试场景：3分类问题，批次大小为2
    num_classes = 3
    batch_size = 2
    
    # 随机生成模型输出logits
    logits = torch.randn(batch_size, num_classes, requires_grad=True)
    # 真实标签（类别索引）
    labels = torch.tensor([0, 1])
    
    # 计算标签平滑交叉熵损失（epsilon=0.1）
    loss = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    
    print("标签平滑交叉熵损失计算成功")
    print(f"输入logits形状: {logits.shape}")
    print(f"输入labels: {labels}")
    print(f"计算得到的损失值: {loss.item()}")

标签平滑交叉熵损失计算成功
输入logits形状: torch.Size([2, 3])
输入labels: tensor([0, 1])
计算得到的损失值: 1.0586960315704346
